In [1]:
# installing required modules
! pip install pandas
! pip install nltk
! pip install scikit-learn
! pip install streamlit
! pip install joblib

In [2]:
### import commands
import pandas as pd
import re
import joblib 
import pickle
import nltk

from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/genaienggkomalgore/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/genaienggkomalgore/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/genaienggkomalgore/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
stopword_list = stopwords.words('english')
print(stopword_list)

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

===================================================================
## Assignment 20: Building & Deploying a Recommendation System
===================================================================


------------------------------------------
# PART 1 — Data Preprocessing
------------------------------------------

## Task 1: Load & Understand Dataset
1. Load the dataset using Pandas.
2. Print:
    - Dataset shape
    - Column names 
    - First 5 rows
    - Other essential Details
3. Identify text column(s) used for recommendations.

Download [dataset](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata) from here.

In [6]:
# ## Task 1: Load & Understand Dataset

path_credits = "Dataset/tmdb_5000_credits.csv"
path_movies = "Dataset/tmdb_5000_movies.csv"


# 1. Load the dataset using Pandas.

credits = pd.read_csv(path_credits)
movies = pd.read_csv(path_movies)

# 2. Print:
#     - Dataset shape
print("*"*100)
print(f"Dataset Shape : \nMovie : {movies.shape} \nCredits : {credits.shape}")
print("*"*100)

#     - Column names 

print(f"Column names : \nMovie : {movies.columns} \nCredits : {credits.columns}")
print("*"*100)
#     - First 5 rows
print(f"First 5 rows : \nMovie : {movies.head()} {"#"*100}\n\nCredits : {credits.head()}")
print("*"*100)
#     - Other essential Details

print(f"Describe : \nMovie : {movies.describe()} {"#"*100}\n\nCredits : {credits.describe()}")
print("*"*100)

print(f"IsNULL : \nMovie : {movies.isnull().sum()} {"#"*100}\n\nCredits : {credits.isnull().sum()}")
print("*"*100)

print(f"Duplicated Values : \nMovie : {movies.duplicated().sum()} {"#"*100}\n\nCredits : {credits.duplicated().sum()}")
print("*"*100)


# 3. Identify text column(s) used for recommendations.

****************************************************************************************************
Dataset Shape : 
Movie : (4803, 20) 
Credits : (4803, 4)
****************************************************************************************************
Column names : 
Movie : Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='str') 
Credits : Index(['movie_id', 'title', 'cast', 'crew'], dtype='str')
****************************************************************************************************
First 5 rows : 
Movie :       budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id"

In [7]:
movies.rename(columns= {'id':'movie_id'},inplace= True)

In [8]:
df = pd.merge(movies , credits , on = 'movie_id')
df.columns

# colunmns we don't  need : 
col_names = ['budget', 'homepage','original_title', 'movie_id', 'keywords','original_language', 'production_companies', 'production_countries', 'release_date','revenue', 'runtime', 'spoken_languages', 'status','title_x', 'vote_average', 'vote_count','popularity'] 
df.drop(columns= col_names , inplace=True)
df = df.dropna()
df.rename(columns= {"title_y":"title"},inplace= True)
# df.info()

In [9]:
df['tagline'].isna().sum()

np.int64(0)

## Task 2: Text Preprocessing for Recommendation
Apply the following steps on the text column:
1. Convert text to lowercase
2. Remove punctuation and special characters
3. Remove stopwords
4. Handle missing values (replace with empty string)
Store cleaned text in a new column: clean_text.


In [10]:
df['tagline'] = df['tagline'].fillna('',inplace=True)




/var/folders/n_/7z9tgm7508z7_wn1z6d0p3fw0000gn/T/ipykernel_40711/1874935130.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['tagline'] = df['tagline'].fillna('',inplace=True)


In [11]:
df['cast'][0]

'[{"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}, {"cast_id": 3, "character": "Neytiri", "credit_id": "52fe48009251416c750ac9cb", "gender": 1, "id": 8691, "name": "Zoe Saldana", "order": 1}, {"cast_id": 25, "character": "Dr. Grace Augustine", "credit_id": "52fe48009251416c750aca39", "gender": 1, "id": 10205, "name": "Sigourney Weaver", "order": 2}, {"cast_id": 4, "character": "Col. Quaritch", "credit_id": "52fe48009251416c750ac9cf", "gender": 2, "id": 32747, "name": "Stephen Lang", "order": 3}, {"cast_id": 5, "character": "Trudy Chacon", "credit_id": "52fe48009251416c750ac9d3", "gender": 1, "id": 17647, "name": "Michelle Rodriguez", "order": 4}, {"cast_id": 8, "character": "Selfridge", "credit_id": "52fe48009251416c750ac9e1", "gender": 2, "id": 1771, "name": "Giovanni Ribisi", "order": 5}, {"cast_id": 7, "character": "Norm Spellman", "credit_id": "52fe48009251416c750ac9dd", "gender": 

In [12]:
import json



In [13]:
def top_5_cast(data):
    result = []
    json_data = json.loads(data)
    for i in range(0,5):
        try:
            result.append(json_data[i]['name'])
        except IndexError:
            break
        

    return " ".join(result)

df['cast'] = df['cast'].apply(lambda data : top_5_cast(data))






In [14]:
# crew
def get_director(data):
    json_data = json.loads(data)
    result = []
    for i in json_data:
        if i['job']== "Director":
            result.append(i['name'])
    return " ".join(result)

df['crew'] = df['crew'].apply(lambda data : get_director(data))


In [15]:
# genres

def get_genres(data):
    json_data = json.loads(data)
    result = []
    for i in json_data:
        result.append(i['name'])
    return " ".join(result)

#get_genres(df['genres'][0])
df['genres'] = df['genres'].apply(lambda data : get_genres(data))


In [16]:
df.head()

,genres,overview,tagline,title,cast,crew
0,Action Adventure Fantasy Science Fiction,"In the 22nd century, a paraplegic Marine is di...",Enter the World of Pandora.,Avatar,Sam Worthington Zoe Saldana Sigourney Weaver S...,James Cameron
1,Adventure Fantasy Action,"Captain Barbossa, long believed to be dead, ha...","At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,Johnny Depp Orlando Bloom Keira Knightley Stel...,Gore Verbinski
2,Action Adventure Crime,A cryptic message from Bond’s past sends him o...,A Plan No One Escapes,Spectre,Daniel Craig Christoph Waltz Léa Seydoux Ralph...,Sam Mendes
3,Action Crime Drama Thriller,Following the death of District Attorney Harve...,The Legend Ends,The Dark Knight Rises,Christian Bale Michael Caine Gary Oldman Anne ...,Christopher Nolan
4,Action Adventure Science Fiction,"John Carter is a war-weary, former military ca...","Lost in our world, found in another.",John Carter,Taylor Kitsch Lynn Collins Samantha Morton Wil...,Andrew Stanton


In [17]:
def stop_word_removal(data):
    result = []
    for i in data.split():
        if i not in stopword_list:
            result.append(i)
    return " ".join(result)



In [18]:
col_names = list(df.columns)

col_names.remove("title")

In [19]:
# Apply the following steps on the text column:



# 1. Convert text to lowercase

df['tagline'] = df['tagline'].apply(lambda data : str(data))

for i in col_names:
    df[i] = df[i].apply(lambda data : data.lower())
    
# 2. Remove punctuation and special characters

for i in col_names:
    df[i] = df[i].apply(lambda data : re.sub(r"[^\w ^\d]"," ",data))
   

# 3. Remove stopwords

for i in col_names:
    df[i] = df[i].apply(lambda data : stop_word_removal(data))
   

# 4. Handle missing values (replace with empty string)

# already done
   
# remove digit

for i in col_names:
    df[i] = df[i].apply(lambda data : re.sub(r'[1234567890]','',data))


# Store cleaned text in a new column: clean_text.
df['clean_text'] = df['crew'] + " " +df['genres'] + " " + df['overview']+ " " +df['tagline'] + " " +df['cast']

In [20]:
df.head()

,genres,overview,tagline,title,cast,crew,clean_text
0,action adventure fantasy science fiction,nd century paraplegic marine dispatched moon p...,enter world pandora,Avatar,sam worthington zoe saldana sigourney weaver s...,james cameron,james cameron action adventure fantasy science...
1,adventure fantasy action,captain barbossa long believed dead come back ...,end world adventure begins,Pirates of the Caribbean: At World's End,johnny depp orlando bloom keira knightley stel...,gore verbinski,gore verbinski adventure fantasy action captai...
2,action adventure crime,cryptic message bond past sends trail uncover ...,plan one escapes,Spectre,daniel craig christoph waltz léa seydoux ralph...,sam mendes,sam mendes action adventure crime cryptic mess...
3,action crime drama thriller,following death district attorney harvey dent ...,legend ends,The Dark Knight Rises,christian bale michael caine gary oldman anne ...,christopher nolan,christopher nolan action crime drama thriller ...
4,action adventure science fiction,john carter war weary former military captain ...,lost world found another,John Carter,taylor kitsch lynn collins samantha morton wil...,andrew stanton,andrew stanton action adventure science fictio...


In [21]:
# drop other collunms

df.drop(columns= col_names , inplace = True)


In [22]:
df.reset_index(inplace=True)

In [23]:
df.head()

,index,title,clean_text
0,0,Avatar,james cameron action adventure fantasy science...
1,1,Pirates of the Caribbean: At World's End,gore verbinski adventure fantasy action captai...
2,2,Spectre,sam mendes action adventure crime cryptic mess...
3,3,The Dark Knight Rises,christopher nolan action crime drama thriller ...
4,4,John Carter,andrew stanton action adventure science fictio...


------------------------------------------
# PART 2 — Text Vectorization
------------------------------------------

## Task 3: Vectorization using TF-IDF
1. Use TfidfVectorizer) to convert text into vectors.
2. Set reasonable parameters:
    - max_features
    - ngram_range
3. Display:
    - Shape of TF-IDF matrix

In [ ]:
# ## Task 3: Vectorization using TF-IDF
# 1. Use TfidfVectorizer) to convert text into vectors.

tidf = TfidfVectorizer( max_features= 3500 , ngram_range=(1,1))

Y = df['title']

X = tidf.fit_transform(df['clean_text'])
result = X.toarray()
# 2. Set reasonable parameters:
#     - max_features
#     - ngram_range
# 3. Display:
#     - Shape of TF-IDF matrix

print(f"Shape of TF-IDF matrix : {result.shape}")

Shape of TF-IDF matrix : (3959, 3500)


KeyboardInterrupt: 

## Task 4: Similarity Computation
1. Compute cosine similarity between all items.
2. Store similarity matrix.
3. Explain briefly why cosine similarity is used.



In [ ]:
# ## Task 4: Similarity Computation
# 1. Compute cosine similarity between all items.

similarity = cosine_similarity(result)

# 2. Store similarity matrix.

print(similarity)

# 3. Explain briefly why cosine similarity is used.

# TO DO

[[1.         0.04584518 0.03527537 ... 0.04087836 0.         0.03716042]
 [0.04584518 1.         0.03502503 ... 0.02244027 0.         0.01869092]
 [0.03527537 0.03502503 1.         ... 0.0125013  0.         0.07713974]
 ...
 [0.04087836 0.02244027 0.0125013  ... 1.         0.         0.00888758]
 [0.         0.         0.         ... 0.         1.         0.00757474]
 [0.03716042 0.01869092 0.07713974 ... 0.00888758 0.00757474 1.        ]]


In [ ]:
tidf.idf_

array([5.18965474, 5.91670347, 6.79909265, ..., 6.71904995, 6.33956033,
       6.98141421], shape=(3500,))

In [ ]:
tidf.get_feature_names_out()

array(['aaron', 'abandoned', 'abducted', ..., 'zone', 'zooey', 'zucker'],
      shape=(3500,), dtype=object)

------------------------------------------

# PART 3 — Recommendation Logic

------------------------------------------

## Task 5: Build Recommendation Function
Create a function:

```def recommend(item_name, top_n=5):```

```returns top N similar items```

Function should:
1. Find index of the selected item
2. Compute similarity scores
3. Sort and return top recommendations
Test with at least 3 different items.
+


In [ ]:
# ## Task 5: Build Recommendation Function
# Create a function:

# ```def recommend(item_name, top_n=5):```

def recommend(item_name, top_n =5):
    result = []
    index = df[df['title']==item_name].index[0]
    responce = similarity[index]
    value = sorted(enumerate(responce), key= lambda x : x[1], reverse= True)[1:top_n+1]
    for i in value:
        result.append(df['title'][i[0]])
        
    


    return result
# ```returns top N similar items```



# Function should:
# 1. Find index of the selected item2. Compute similarity scores
# 3. Sort and return top recommendations
# Test with at least 3 different items.
print(recommend(df['title'][5]))
print(recommend(df['title'][5],1))
print(recommend(df['title'][5],3))

['Spider-Man 2', 'Spider-Man', 'Arachnophobia', 'The Amazing Spider-Man', 'The Amazing Spider-Man 2']
['Spider-Man 2']
['Spider-Man 2', 'Spider-Man', 'Arachnophobia']


In [ ]:
# Creating file

joblib.dump(similarity,'similarities.joblib',compress=3)

# Save data to a binary file
with open("dataset.dat", "wb") as f:
    pickle.dump(df, f)

# 

------------------------------------------
# PART 4 — Simple App Interface
------------------------------------------

## Task 6: Build Ul using Streamlit
1. Create a simple interface:
• Dropdown to select item
• Button to generate recommendations
2. Display recommended items clearly.
You may use:
Streamlit (preferred)

In [ ]:
# All code is in main.py file

import os
path = os.getcwd()+"/app.py"

os.system(f"streamlit run {path}")

2026-05-08 18:54:55.621 Uvicorn server started on 0.0.0.0:8502



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://192.168.29.87:8502

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
Aliens
Apollo 18
Colombiana
Aliens vs Predator: Requiem
Guardians of the Galaxy
Aliens
Apollo 18
Colombiana
Aliens vs Predator: Requiem
Guardians of the Galaxy
Aliens
Apollo 18
Colombiana
Aliens vs Predator: Requiem
Guardians of the Galaxy
Aliens
Apollo 18
Colombiana
Aliens vs Predator: Requiem
Guardians of the Galaxy
Aliens
Apollo 18
Colombiana
Aliens vs Predator: Requiem
Guardians of the Galaxy
Aliens<br>Apollo 18<br>Colombiana<br>Aliens vs Predator: Requiem<br>Guardians of the Galaxy
Aliens<br>Apollo 18<br>Colombiana<br>Aliens vs Predator: Requiem<br>Guardians of the Galaxy
['Aliens', 'Apollo 18', 'Colombiana', 'Aliens vs Predator: Requiem', 'Guardians of the Galaxy']
['Aliens', 'Apollo 18', 'Colombiana', 'Aliens vs Predator: Requie

2026-05-08 18:56:50.120 Script compilation error
Traceback (most recent call last):
  File "/Users/genaienggkomalgore/Documents/Assignment/Assignment20/.venv/lib/python3.13/site-packages/streamlit/runtime/scriptrunner/script_runner.py", line 591, in _run_script
    code = self._script_cache.get_bytecode(script_path)
  File "/Users/genaienggkomalgore/Documents/Assignment/Assignment20/.venv/lib/python3.13/site-packages/streamlit/runtime/scriptrunner/script_cache.py", line 72, in get_bytecode
    filebody = magic.add_magic(filebody, script_path)
  File "/Users/genaienggkomalgore/Documents/Assignment/Assignment20/.venv/lib/python3.13/site-packages/streamlit/runtime/scriptrunner/magic.py", line 45, in add_magic
    tree = ast.parse(code, script_path, "exec")
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/ast.py", line 50, in parse
    return compile(source, filename, mode, flags,
                   _feature_version=feature_version, optimize=optimize)
  File "/User

['Aliens', 'Apollo 18', 'Colombiana', 'Aliens vs Predator: Requiem', 'Guardians of the Galaxy']
['Aliens', 'Apollo 18', 'Colombiana', 'Aliens vs Predator: Requiem', 'Guardians of the Galaxy']
['Aliens', 'Apollo 18', 'Colombiana', 'Aliens vs Predator: Requiem', 'Guardians of the Galaxy']
['Aliens', 'Apollo 18', 'Colombiana', 'Aliens vs Predator: Requiem', 'Guardians of the Galaxy']
['Aliens', 'Apollo 18', 'Colombiana', 'Aliens vs Predator: Requiem', 'Guardians of the Galaxy']
['Aliens', 'Apollo 18', 'Colombiana', 'Aliens vs Predator: Requiem', 'Guardians of the Galaxy']
['Aliens', 'Apollo 18', 'Colombiana', 'Aliens vs Predator: Requiem', 'Guardians of the Galaxy']
['Aliens', 'Apollo 18', 'Colombiana', 'Aliens vs Predator: Requiem', 'Guardians of the Galaxy']
['Aliens', 'Apollo 18', 'Colombiana', 'Aliens vs Predator: Requiem', 'Guardians of the Galaxy']
['Aliens', 'Apollo 18', 'Colombiana', 'Aliens vs Predator: Requiem', 'Guardians of the Galaxy']
  Stopping...


0

## PART 5 — Version Control with Git & GitHub

Task 7: Git & GitHub Setup
1. Initialize a Git repository.
2. Create a GitHub repository.
3. Push your project code to GitHub.
Repository must include:
app.py
requirements.txt
README.md


# DONE


------------------------------------------
# PART 6 - Deployment on Render
------------------------------------------

## Task 8: Deploy Application on Render
1. Create a Render account.
2. Connect Render with GitHub.
3. Select your repository.
4. Configure:
    - Build command
    - Start command

# 5. Deploy the application.
Task 9: Final Validation
1. Test deployed app link.
2. Ensure recommendations work correctly.
3. Include deployed URL in submission.